# Fetching data at warp speed

In [5]:
# 1. Install dependencies
!pip install dukascopy-python pandas pyarrow joblib -q

import os
import time
import pandas as pd
import dukascopy_python as dp
from datetime import datetime
from pathlib import Path
from joblib import Parallel, delayed

# --- CONFIGURATION ---
# NOTE: If you want to keep files permanently, mount Google Drive:
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = Path("/content/drive/MyDrive/GITHUB-COPILOT/stk-mat2011/data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PAIRS = ["EUR/USD"]
MONTHS = [
    "202501", "202502", "202503", "202504", "202505", "202506",
    "202507", "202508", "202509", "202510", "202511", "202512"]

# PARALLELISM: Use -1 to use all cores, or 4-8 to avoid getting rate-limited
N_JOBS = 4 

# --- HELPER FUNCTIONS ---
def get_month_range(yyyymm: str):
    year, month = int(yyyymm[:4]), int(yyyymm[4:6])
    start = datetime(year, month, 1)
    end = datetime(year + 1, 1, 1) if month == 12 else datetime(year, month + 1, 1)
    return start, end

def process_job(pair, yyyymm, compression="snappy", max_retries=3):
    """Function to be executed in parallel with existence check and retries"""
    symbol = pair.replace("/", "").lower()
    
    # 1. EXISTENCE CHECK: Define file paths first
    out_name_bid = f"{symbol}_dukascopy_bid_{yyyymm}.parquet"
    out_name_ask = f"{symbol}_dukascopy_ask_{yyyymm}.parquet"
    out_path_bid = OUT_DIR / out_name_bid
    out_path_ask = OUT_DIR / out_name_ask

    # If both files already exist, skip the download completely
    if out_path_bid.exists() and out_path_ask.exists():
        return f"⏭️ Skipped (Already Exists): {pair} {yyyymm}"

    start, end = get_month_range(yyyymm)
    
    # 2. RETRY LOGIC: Handle Dukascopy connection drops
    for attempt in range(max_retries):
        try:
            df = dp.fetch(
                instrument=pair,
                interval=dp.INTERVAL_TICK,
                offer_side=dp.OFFER_SIDE_BID, 
                start=start,
                end=end,
            )

            if df is None or len(df) == 0:
                return f"⚠️ Empty: {pair} {yyyymm}"

            results_stats = []
            for side, price_col, vol_col in [("bid", "bidPrice", "bidVolume"), ("ask", "askPrice", "askVolume")]:
                out_name = out_name_bid if side == "bid" else out_name_ask
                out_path = out_path_bid if side == "bid" else out_path_ask
                
                pd.DataFrame({
                    "datetime": df.index,
                    "symbol": symbol.upper(),
                    "price_type": side.upper(),
                    "price": df[price_col].values,
                    "volume": df[vol_col].values,
                }).to_parquet(out_path, engine="pyarrow", compression=compression, index=False)
                
                results_stats.append(f"{out_name} ({len(df):,} rows)")
                
            return f"✅ Success: {pair} {yyyymm} -> " + " & ".join(results_stats)
        
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(5) # Wait 5 seconds before retrying
                continue
            return f"❌ FAILED: {pair} {yyyymm} | Error after {max_retries} attempts: {str(e)}"

# --- EXECUTION ---
jobs = [(p, m) for p in PAIRS for m in MONTHS]
print(f"🚀 Starting Parallel Download of {len(jobs)} jobs using {N_JOBS} workers...\n")

t0 = time.time()
results = Parallel(n_jobs=N_JOBS, backend="threading")(
    delayed(process_job)(p, m) for p, m in jobs
)

for r in results:
    print(r)

elapsed = time.time() - t0
print(f"\n✨ All done! Finished in {elapsed:.1f}s")
print(f"📁 Files saved to: {OUT_DIR.absolute()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Starting Parallel Download of 12 jobs using 4 workers...



INFO:DUKASCRIPT:current timestamp :2025-01-02T09:39:00.503000
INFO:DUKASCRIPT:current timestamp :2025-02-03T01:26:53.687000
INFO:DUKASCRIPT:current timestamp :2025-03-03T06:34:49.775000
INFO:DUKASCRIPT:current timestamp :2025-04-01T09:30:32.394000
INFO:DUKASCRIPT:current timestamp :2025-01-02T14:33:12.837000
INFO:DUKASCRIPT:current timestamp :2025-02-03T06:24:17.177000
INFO:DUKASCRIPT:current timestamp :2025-04-01T14:37:30.740000
INFO:DUKASCRIPT:current timestamp :2025-01-02T17:14:34.329000
INFO:DUKASCRIPT:current timestamp :2025-03-03T11:56:10.770000
INFO:DUKASCRIPT:current timestamp :2025-02-03T10:45:11.409000
INFO:DUKASCRIPT:current timestamp :2025-04-02T01:18:13.757000
INFO:DUKASCRIPT:current timestamp :2025-01-03T08:07:14.322000
INFO:DUKASCRIPT:current timestamp :2025-03-03T15:56:50.883000
INFO:DUKASCRIPT:current timestamp :2025-04-02T11:30:12.232000
INFO:DUKASCRIPT:current timestamp :2025-01-03T15:10:17.742000
INFO:DUKASCRIPT:current timestamp :2025-04-02T16:16:08.424000
INFO:DUK

✅ Success: EUR/USD 202501 -> eurusd_dukascopy_bid_202501.parquet (2,283,240 rows) & eurusd_dukascopy_ask_202501.parquet (2,283,240 rows)
✅ Success: EUR/USD 202502 -> eurusd_dukascopy_bid_202502.parquet (2,053,416 rows) & eurusd_dukascopy_ask_202502.parquet (2,053,416 rows)
✅ Success: EUR/USD 202503 -> eurusd_dukascopy_bid_202503.parquet (2,347,511 rows) & eurusd_dukascopy_ask_202503.parquet (2,347,511 rows)
✅ Success: EUR/USD 202504 -> eurusd_dukascopy_bid_202504.parquet (3,916,482 rows) & eurusd_dukascopy_ask_202504.parquet (3,916,482 rows)
✅ Success: EUR/USD 202505 -> eurusd_dukascopy_bid_202505.parquet (2,423,223 rows) & eurusd_dukascopy_ask_202505.parquet (2,423,223 rows)
✅ Success: EUR/USD 202506 -> eurusd_dukascopy_bid_202506.parquet (2,033,530 rows) & eurusd_dukascopy_ask_202506.parquet (2,033,530 rows)
✅ Success: EUR/USD 202507 -> eurusd_dukascopy_bid_202507.parquet (1,558,153 rows) & eurusd_dukascopy_ask_202507.parquet (1,558,153 rows)
✅ Success: EUR/USD 202508 -> eurusd_dukas